***

Profesor: Gonzalo A. Ruz, PhD

Curso: Deep Learning

***

# Clasificación, Regresión y Regularización en Keras

En este notebook veremos cómo aplicar redes neuronales densas a distintos tipos de problemas:

- clasificación binaria,
- clasificación multiclase,
- regresión,
- regularización para mejorar generalización.

## Objetivo

Al finalizar este notebook deberías poder:

- construir una red neuronal densa en Keras,
- elegir la capa de salida adecuada según el problema,
- seleccionar una función de pérdida coherente,
- entrenar y validar un modelo,
- identificar señales de overfitting,
- aplicar estrategias básicas de regularización.

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers
import numpy as np
import matplotlib.pyplot as plt

tf.random.set_seed(42)
np.random.seed(42)

print("TensorFlow:", tf.__version__)
print("Keras:", keras.__version__)

## Hoja de ruta

1. Clasificación binaria
2. Clasificación multiclase
3. Regresión
4. Regularización

## 1. Clasificación binaria: reseñas de películas ([IMDB](https://www.imdb.com/))

Queremos predecir si una reseña es positiva o negativa.

Este es un problema de **clasificación binaria**, por lo tanto usaremos:

- una sola neurona de salida,
- activación `sigmoid`,
- pérdida `binary_crossentropy`.

In [ ]:
from tensorflow.keras.datasets import imdb

(train_data, train_labels), (test_data, test_labels) = imdb.load_data(num_words=10000)

print("Train samples:", len(train_data))
print("Test samples:", len(test_data))

In [ ]:
print(train_data[0][:20])
print("Label:", train_labels[0])

In [ ]:
word_index = imdb.get_word_index()
reverse_word_index = dict([(value, key) for (key, value) in word_index.items()])
print("Original review: ", ' '.join([reverse_word_index.get(i - 3, '?') for i in train_data[0]][0:20]))

### Preprocesamiento

Cada reseña está codificada como una secuencia de [índices de palabras](https://storage.googleapis.com/tensorflow/tf-keras-datasets/imdb_word_index.json).

Como usaremos una red densa, transformaremos cada reseña en un vector binario de tamaño 10.000.

In [ ]:
def vectorize_sequences(sequences, dimension=10000):
    results = np.zeros((len(sequences), dimension))
    for i, sequence in enumerate(sequences):
        results[i, sequence] = 1.
    return results

x_train = vectorize_sequences(train_data)
x_test = vectorize_sequences(test_data)

y_train = np.asarray(train_labels).astype("float32")
y_test = np.asarray(test_labels).astype("float32")

print(x_train.shape)
print(x_test.shape)

### Construcción del modelo

Usaremos una red pequeña con dos capas ocultas y una salida sigmoide.

In [ ]:
binary_model = keras.Sequential([
    keras.Input(shape=(10000,)),
    layers.Dense(16, activation="relu"),
    layers.Dense(16, activation="relu"),
    layers.Dense(1, activation="sigmoid")
])

binary_model.summary()

In [ ]:
binary_model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

In [ ]:
x_val = x_train[:10000]
partial_x_train = x_train[10000:]

y_val = y_train[:10000]
partial_y_train = y_train[10000:]

In [ ]:
binary_history = binary_model.fit(
    partial_x_train,
    partial_y_train,
    epochs=20,
    batch_size=512,
    validation_data=(x_val, y_val),
    verbose=1
)

In [ ]:
history_dict = binary_history.history
loss_values = history_dict["loss"]
val_loss_values = history_dict["val_loss"]
epochs = range(1, len(loss_values) + 1)

plt.figure(figsize=(6,4))
plt.plot(epochs, loss_values, label="Train loss")
plt.plot(epochs, val_loss_values, label="Val loss")
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.legend()
plt.show()

In [ ]:
acc_values = history_dict["accuracy"]
val_acc_values = history_dict["val_accuracy"]

plt.figure(figsize=(6,4))
plt.plot(epochs, acc_values, label="Train accuracy")
plt.plot(epochs, val_acc_values, label="Val accuracy")
plt.xlabel("Epochs")
plt.ylabel("Accuracy")
plt.legend()
plt.show()

### Preguntas para discutir

- ¿En qué momento comienza el sobreajuste?
- ¿Qué observamos al comparar train y validation?

In [ ]:
binary_results = binary_model.evaluate(x_test, y_test, verbose=0)
print("Test loss:", binary_results[0])
print("Test accuracy:", binary_results[1])

### Predicciones
Por curiosidad, veamos algunas predicciones:

In [ ]:
predictions = binary_model.predict(x_test)
print("Review 0: ", ' '.join([reverse_word_index.get(i - 3, '?') for i in test_data[0]]))
print("Predicted positiveness: ", predictions[0])
print("\nReview 16: ", ' '.join([reverse_word_index.get(i - 3, '?') for i in test_data[16]]))
print("Predicted positiveness: ", predictions[16])

### Qué recordar

- problema: clasificación binaria,
- salida: 1 neurona,
- activación final: `sigmoid`,
- pérdida: `binary_crossentropy`.

## 2. Clasificación multiclase: temas de noticias (Reuters)

Ahora queremos clasificar cada noticia en una entre varias categorías.

Este es un problema de **clasificación multiclase**, por lo tanto usaremos:

- una neurona por clase,
- activación `softmax`,
- pérdida `categorical_crossentropy`.

In [ ]:
from tensorflow.keras.datasets import reuters

(train_data, train_labels), (test_data, test_labels) = reuters.load_data(num_words=10000)

print("Train samples:", len(train_data))
print("Test samples:", len(test_data))

In [ ]:
x_train = vectorize_sequences(train_data)
x_test = vectorize_sequences(test_data)

In [ ]:
from tensorflow.keras.utils import to_categorical

y_train = to_categorical(train_labels)
y_test = to_categorical(test_labels)

print(y_train.shape)

In [ ]:
multiclass_model = keras.Sequential([
    keras.Input(shape=(10000,)),
    layers.Dense(64, activation="relu"),
    layers.Dense(64, activation="relu"),
    layers.Dense(46, activation="softmax")
])

multiclass_model.summary()

In [ ]:
multiclass_model.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

In [ ]:
x_val = x_train[:1000]
partial_x_train = x_train[1000:]

y_val = y_train[:1000]
partial_y_train = y_train[1000:]

In [ ]:
multiclass_history = multiclass_model.fit(
    partial_x_train,
    partial_y_train,
    epochs=12,
    batch_size=512,
    validation_data=(x_val, y_val),
    verbose=1
)

In [ ]:
history_dict = multiclass_history.history
loss = history_dict["loss"]
val_loss = history_dict["val_loss"]
acc = history_dict["accuracy"]
val_acc = history_dict["val_accuracy"]
epochs = range(1, len(loss) + 1)

plt.figure(figsize=(6,4))
plt.plot(epochs, loss, label="Train loss")
plt.plot(epochs, val_loss, label="Val loss")
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.legend()
plt.show()

plt.figure(figsize=(6,4))
plt.plot(epochs, acc, label="Train accuracy")
plt.plot(epochs, val_acc, label="Val accuracy")
plt.xlabel("Epochs")
plt.ylabel("Accuracy")
plt.legend()
plt.show()

In [ ]:
multiclass_results = multiclass_model.evaluate(x_test, y_test, verbose=0)
print("Test loss:", multiclass_results[0])
print("Test accuracy:", multiclass_results[1])

### Qué cambia respecto al caso binario

- ahora hay muchas clases,
- la salida tiene una neurona por clase,
- usamos `softmax`,
- usamos `categorical_crossentropy`.

## 3. Regresión con redes neuronales

En clasificación, predecimos una clase.

En regresión, queremos predecir un valor numérico continuo.

Por eso usaremos:

- una capa de salida con 1 neurona,
- activación lineal,
- pérdida `mse`,
- métrica `mae`.

In [ ]:
from tensorflow.keras.datasets import boston_housing

(train_data, train_targets), (test_data, test_targets) = boston_housing.load_data()

print("Train data shape:", train_data.shape)
print("Test data shape:", test_data.shape)

### Preprocesamiento

En problemas tabulares de regresión es importante normalizar o estandarizar las variables de entrada.

In [ ]:
mean = train_data.mean(axis=0)
train_data -= mean
test_data -= mean

std = train_data.std(axis=0)
train_data /= std
test_data /= std

In [ ]:
regression_model = keras.Sequential([
    keras.Input(shape=(train_data.shape[1],)),
    layers.Dense(64, activation="relu"),
    layers.Dense(64, activation="relu"),
    layers.Dense(1)
])

regression_model.summary()

### Observación importante

La capa de salida no usa `sigmoid` ni `softmax`, porque aquí no queremos probabilidades ni clases, sino un valor real.

In [ ]:
regression_model.compile(
    optimizer="adam",
    loss="mse",
    metrics=["mae"]
)

In [ ]:
regression_history = regression_model.fit(
    train_data,
    train_targets,
    epochs=40,
    batch_size=16,
    validation_split=0.2,
    verbose=1
)

In [ ]:
history_dict = regression_history.history
loss = history_dict["loss"]
val_loss = history_dict["val_loss"]
mae = history_dict["mae"]
val_mae = history_dict["val_mae"]
epochs = range(1, len(loss) + 1)

plt.figure(figsize=(6,4))
plt.plot(epochs, loss, label="Train loss")
plt.plot(epochs, val_loss, label="Val loss")
plt.xlabel("Epochs")
plt.ylabel("MSE loss")
plt.legend()
plt.show()

plt.figure(figsize=(6,4))
plt.plot(epochs, mae, label="Train MAE")
plt.plot(epochs, val_mae, label="Val MAE")
plt.xlabel("Epochs")
plt.ylabel("MAE")
plt.legend()
plt.show()

In [ ]:
test_mse, test_mae = regression_model.evaluate(test_data, test_targets, verbose=0)
print("Test MSE:", test_mse)
print("Test MAE:", test_mae)

In [ ]:
preds = regression_model.predict(test_data[:5], verbose=0).flatten()

for i in range(5):
    print(f"Real: {test_targets[i]:.2f} | Predicha: {preds[i]:.2f}")

### Qué recordar

- problema: regresión,
- salida: 1 neurona,
- activación final: lineal,
- pérdida: `mse`,
- métrica útil: `mae`.

## 4. Regularización

Aumentar la capacidad del modelo puede mejorar el ajuste en entrenamiento, pero también aumentar el riesgo de sobreajuste.

Probemos algunas estrategias básicas de regularización usando nuevamente IMDB.

In [ ]:
(train_data, train_labels), (test_data, test_labels) = imdb.load_data(num_words=10000)

x_train = vectorize_sequences(train_data)
x_test = vectorize_sequences(test_data)

y_train = np.asarray(train_labels).astype("float32")
y_test = np.asarray(test_labels).astype("float32")

x_val = x_train[:10000]
partial_x_train = x_train[10000:]

y_val = y_train[:10000]
partial_y_train = y_train[10000:]

In [ ]:
original_model = keras.Sequential([
    keras.Input(shape=(10000,)),
    layers.Dense(16, activation="relu"),
    layers.Dense(16, activation="relu"),
    layers.Dense(1, activation="sigmoid")
])

original_model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

original_hist = original_model.fit(
    partial_x_train,
    partial_y_train,
    epochs=20,
    batch_size=512,
    validation_data=(x_val, y_val),
    verbose=0
)

In [ ]:
smaller_model = keras.Sequential([
    keras.Input(shape=(10000,)),
    layers.Dense(4, activation="relu"),
    layers.Dense(4, activation="relu"),
    layers.Dense(1, activation="sigmoid")
])

smaller_model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

smaller_hist = smaller_model.fit(
    partial_x_train,
    partial_y_train,
    epochs=20,
    batch_size=512,
    validation_data=(x_val, y_val),
    verbose=0
)

In [ ]:
original_val_loss = original_hist.history["val_loss"]
smaller_val_loss = smaller_hist.history["val_loss"]
epochs = range(1, len(original_val_loss) + 1)

plt.figure(figsize=(6,4))
plt.plot(epochs, original_val_loss, label="Original model")
plt.plot(epochs, smaller_val_loss, label="Smaller model")
plt.xlabel("Epochs")
plt.ylabel("Validation loss")
plt.legend()
plt.show()

### L2

In [ ]:
l2_model = keras.Sequential([
    keras.Input(shape=(10000,)),
    layers.Dense(16, kernel_regularizer=regularizers.l2(0.001), activation="relu"),
    layers.Dense(16, kernel_regularizer=regularizers.l2(0.001), activation="relu"),
    layers.Dense(1, activation="sigmoid")
])

l2_model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

l2_hist = l2_model.fit(
    partial_x_train,
    partial_y_train,
    epochs=20,
    batch_size=512,
    validation_data=(x_val, y_val),
    verbose=0
)

In [ ]:
l2_val_loss = l2_hist.history["val_loss"]

plt.figure(figsize=(6,4))
plt.plot(epochs, original_val_loss, label="Original model")
plt.plot(epochs, l2_val_loss, label="L2 model")
plt.xlabel("Epochs")
plt.ylabel("Validation loss")
plt.legend()
plt.show()

### Dropout
* Una de las técnicas de regularización más eficaces y utilizadas
* Establecer aleatoriamente una cantidad de salidas de la capa en 0
* Idea: romper patrones aprendidos accidentales no significativos
* *Dropout rate*: fracción de las salidas que se ponen a cero
    - Usualmente entre 0.2 y 0.5
* En el momento de la prueba, no se descarta nada, pero los valores de salida se reducen según la tasa de dropout.
* En Keras: agrega capas `Dropout` entre las capas normales

![](https://drive.google.com/uc?id=1oLYMwj7ars6bVWUxAgsV4tWMsvjOgdQT)
[Link del paper](https://jmlr.org/papers/volume15/srivastava14a/srivastava14a.pdf)

In [ ]:
dropout_model = keras.Sequential([
    keras.Input(shape=(10000,)),
    layers.Dense(16, activation="relu"),
    layers.Dropout(0.5),
    layers.Dense(16, activation="relu"),
    layers.Dropout(0.5),
    layers.Dense(1, activation="sigmoid")
])

dropout_model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

dropout_hist = dropout_model.fit(
    partial_x_train,
    partial_y_train,
    epochs=20,
    batch_size=512,
    validation_data=(x_val, y_val),
    verbose=0
)

In [ ]:
dropout_val_loss = dropout_hist.history["val_loss"]

plt.figure(figsize=(6,4))
plt.plot(epochs, original_val_loss, label="Original model")
plt.plot(epochs, dropout_val_loss, label="Dropout model")
plt.xlabel("Epochs")
plt.ylabel("Validation loss")
plt.legend()
plt.show()

### Early stopping

Otra estrategia simple consiste en detener el entrenamiento cuando la pérdida de validación deja de mejorar.

In [ ]:
early_stop = keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=2,
    restore_best_weights=True
)

In [ ]:
early_model = keras.Sequential([
    keras.Input(shape=(10000,)),
    layers.Dense(16, activation="relu"),
    layers.Dense(16, activation="relu"),
    layers.Dense(1, activation="sigmoid")
])

early_model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

early_hist = early_model.fit(
    partial_x_train,
    partial_y_train,
    epochs=20,
    batch_size=512,
    validation_data=(x_val, y_val),
    callbacks=[early_stop],
    verbose=0
)

In [ ]:
early_results = early_model.evaluate(x_test, y_test, verbose=0)
print("Test loss:", early_results[0])
print("Test accuracy:", early_results[1])

## Resumen de regularización

Estrategias vistas:

- usar una red más pequeña,
- regularización L2,
- dropout,
- early stopping.

No existe una única receta universal. La elección depende del problema, de los datos y de la capacidad del modelo.

## Resumen final

En este notebook vimos cómo cambian la arquitectura, la activación final, la pérdida y las métricas según el tipo de problema:

- clasificación binaria,
- clasificación multiclase,
- regresión.

También vimos que entrenar bien no basta: necesitamos controlar la generalización mediante validación y regularización.